<a href="https://colab.research.google.com/github/shashankcm/RAG_LANGCHAIN_LLM_Projects/blob/main/gradio_image_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip3 install gradio
!pip3 install transformers
!pip3 install torch

In [ ]:
import gradio as gr
import torch
import requests

from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
from torchvision import transforms

#processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
#model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large")

model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', pretrained=True)
model.eval()

# Download human-readable labels for ImageNet
response = requests.get("https://git.io/JJkYN")
labels = [l.strip() for l in response.text.split("\n") if l.strip()]


def generate_caption(image):
  inputs = processor(image, return_tensors="pt")
  outputs = model.generate(**inputs)
  return processor.decode(outputs[0], skip_special_tokens=True)

def caption_image(image):
  """Takes a PIL Image as input and returns Caption as output"""
  try:
    caption = generate_caption(image)
  except Exception as e:
    return f"Error: {e}"

image_caption = gr.Interface(fn=caption_image, inputs=gr.Image(type='pil'), outputs="text", title="Image Captioning with BLIP", description="Upload an image to generate a caption")

#image_caption.launch(server_name="127.0.0.1", server_port=7860)

# Define Image preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def predict(inp):
    # preprocess image
    inp = transform(inp).unsqueeze(0)
    # ensure model runs in inference mode
    with torch.no_grad():
        prediction = torch.nn.functional.softmax(model(inp)[0], dim=0)
    # map predictions to labels
    confidences = {
        labels[i]: float(prediction[i])
        for i in range(len(labels))
    }
    return confidences

image_classification = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs=gr.Label(num_top_classes=3),
    title="Image Classification with ResNet18",
    examples=["/content/images/lion-805084_1920-c62a5582169c4bae82553d9a21c1a0bb.jpg",
              "/content/images/animal-4917802_640.jpg"
             ]
)

image_classification.launch(server_name="127.0.0.1", server_port=7862)


